# CSIRO Biomass - Submission Inference

Inference-only notebook for image checkpoints. Attach a Kaggle dataset containing `checkpoints/fold*.pt`; this notebook writes `/kaggle/working/submission.csv`.

In [ ]:
import os, sys, subprocess, pkgutil
from pathlib import Path
import numpy as np
import pandas as pd
if pkgutil.find_loader('timm') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm'])
import cv2, torch, timm
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast

DATA_DIR = Path('/kaggle/input/csiro-biomass')
INPUT_ROOT = Path('/kaggle/input')
OUT_DIR = Path('/kaggle/working')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda': raise RuntimeError('GPU is required for inference policy.')
TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
BATCH_SIZE = int(os.getenv('BATCH_SIZE', '32'))

In [ ]:
class InferenceModel(nn.Module):
    def __init__(self, backbone, n_targets=5, multitask=True):
        super().__init__(); self.multitask = multitask
        self.encoder = timm.create_model(backbone, pretrained=False, num_classes=0, global_pool='avg')
        dim = self.encoder.num_features
        if multitask:
            self.body = nn.Sequential(nn.LayerNorm(dim), nn.Dropout(0.15), nn.Linear(dim, 512), nn.GELU(), nn.Dropout(0.1))
            self.biomass_head = nn.Linear(512, n_targets); self.aux_head = nn.Linear(512, 2)
        else:
            self.head = nn.Sequential(nn.LayerNorm(dim), nn.Dropout(0.15), nn.Linear(dim, 512), nn.GELU(), nn.Linear(512, n_targets))
    def forward(self, x):
        z = self.encoder(x)
        if self.multitask:
            return self.biomass_head(self.body(z))
        return self.head(z)

ckpts = sorted(INPUT_ROOT.glob('**/checkpoints/fold*.pt')) + sorted(INPUT_ROOT.glob('**/fold*.pt'))
print([str(p) for p in ckpts])
if not ckpts: raise RuntimeError('No fold checkpoints found under /kaggle/input.')

In [ ]:
class TestDataset(Dataset):
    def __init__(self, df, image_size): self.df = df.reset_index(drop=True); self.image_size = image_size
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(str(DATA_DIR / row.image_path)); image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.image_size, self.image_size)).astype(np.float32) / 255.0
        image = (image - np.array([0.485,0.456,0.406], np.float32)) / np.array([0.229,0.224,0.225], np.float32)
        return torch.from_numpy(image.transpose(2,0,1)).float()

test_long = pd.read_csv(DATA_DIR / 'test.csv')
test_images = test_long[['image_path']].drop_duplicates().reset_index(drop=True)
pred_sum = None
for ckpt_path in ckpts:
    ckpt = torch.load(ckpt_path, map_location='cpu')
    backbone = ckpt.get('backbone', 'convnextv2_tiny.fcmae_ft_in22k_in1k')
    image_size = int(ckpt.get('image_size', 384))
    multitask = 'biomass_head.weight' in ckpt['model']
    model = InferenceModel(backbone, multitask=multitask).to(DEVICE)
    model.load_state_dict(ckpt['model'], strict=True); model.eval()
    dl = DataLoader(TestDataset(test_images, image_size), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    preds = []
    with torch.no_grad():
        for x in dl:
            with autocast(): preds.append(model(x.to(DEVICE)).float().cpu().numpy())
    preds = np.concatenate(preds)
    pred_sum = preds if pred_sum is None else pred_sum + preds
pred = pred_sum / len(ckpts)

In [ ]:
pred_wide = test_images.copy()
for i, t in enumerate(TARGETS): pred_wide[t] = pred[:, i]
pred_long = pred_wide.melt(id_vars='image_path', value_vars=TARGETS, var_name='target_name', value_name='target')
sub = test_long[['sample_id', 'image_path', 'target_name']].merge(pred_long, on=['image_path','target_name'], how='left')[['sample_id','target']]
sub['target'] = sub['target'].clip(lower=0)
sub.to_csv(OUT_DIR / 'submission.csv', index=False)
display(sub.head())